# TypeSafe Jev decisions with temporal history

This notebook uses a **hand-written fixture**, not live model output. No API keys, network calls or graph writes are needed. It annotates an offline warehouse replay; it does not control hardware. Run locally for live inference: see `docs/JEV.md`.


In [ ]:
import json
from pathlib import Path
from chronograph_connectors.jev import jev_decision

folder = Path("examples/jev") if Path("examples/jev/request.json").exists() else Path(".")
request = json.loads((folder / "request.json").read_text())
response = json.loads((folder / "response.fixture.json").read_text())


In [ ]:
record = jev_decision(response, request=request, src="9007199254740993",
    dst="9007199254740994", timestamp_us="1700000000000000", mode="fixture")
assert record["fields"]["mode"] == "fixture"
assert record["assets"] == {}  # Raw inputs stay out unless explicitly attached.
record


## Inspect the three decision types

The selected choice, full distributions and rubric remain available when replaying a saved record. These fabricated numbers demonstrate storage, not model quality.


In [ ]:
answers = record["fields"]["answers"]
assert answers["route"]["type"] == "choice"
assert answers["obstructed"]["type"] == "noul"
assert answers["review_priority"]["type"] == "score"
[(name, answer) for name, answer in answers.items()]


## Connect a real graph

Run the CLI locally with `--write --init --attach-inputs` to apply the Jev migration, upload both JSON attachments and read back a durable record. The example uses `CHRONOGRAPH_URL` and a private `CHRONOGRAPH_TOKEN_FILE`. Add `--live` and configure `TYPESAFE_API_KEY` to call TypeSafe. Never put secrets in saved notebook cells or output. See the included documentation for exact setup and limitations.
